# 06. Detect objects over a folder of images

This loops over **image files**, not a video or camera. Start with [notebook 05](05_tiny-yolo-image.ipynb) to see each computation stage separately.

This **Tinier-YOLO** model detects the 20 PASCAL VOC classes: person; bird, cat, cow, dog, horse, sheep; aeroplane, bicycle, boat, bus, car, motorbike, train; bottle, chair, dining table, potted plant, sofa and TV/monitor. It does not have a traffic-sign class.

![Tinier-YOLO topology](Tinier-YOLO-topology.svg)

The first and last convolutions run on the **ARM CPU**, intermediate layers on the **FPGA** (1-bit weights, 3-bit activations), and detection decoding/drawing in native **Darknet on the CPU**. Run cells in order. Shut down other FPGA kernels first. The installed package must already have its unused `Xlnk` imports removed. No model download or package reinstall is needed.

## 1. Check the installed model, Darknet and image files

Run this cell. It resolves the installed package automatically and checks the existing `/opt/darknet` library, VOC labels and drawing assets **before** initializing the FPGA. Upload extra JPG/PNG/BMP files into **yoloimages** beside the notebook. Only image files are selected.

The custom Darknet library installed by this repository is required; an unrelated pip package named `darknet` is not a replacement.

In [ ]:
import sys
import gc
import random
import time
import ctypes
import tempfile
from pathlib import Path

import numpy as np
from PIL import Image
from matplotlib import pyplot as plt
from IPython.display import display
import qnn
from qnn import TinierYolo, utils

QNN_PARAM_DIR = Path(qnn.__file__).resolve().parent / "params"
NOTEBOOK_DIR = Path.cwd()
if not (NOTEBOOK_DIR / "yoloimages").is_dir():
    NOTEBOOK_DIR = Path("/home/xilinx/jupyter_notebooks/qnn")
IMAGE_DIR = NOTEBOOK_DIR / "yoloimages"
DARKNET_DIR = Path("/opt/darknet")
VOC_NAMES = DARKNET_DIR / "data/voc.names"
required_files = [QNN_PARAM_DIR / name for name in (
    "tinier-yolo-layers.json", "tinier-yolo-conv0-W.npy", "tinier-yolo-conv8-W.npy",
    "tinier-yolo-conv0-bias.npy", "tinier-yolo-conv8-bias.npy",
    "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg")]
required_files += [DARKNET_DIR / "python/darknet.py", DARKNET_DIR / "libdarknet.so", VOC_NAMES]
# Darknet loads all printable-character PNGs at each size when drawing labels.
required_files += [DARKNET_DIR / "data/labels" / "{}_{}.png".format(char, size)
                   for size in range(8) for char in range(32, 127)]
missing = [str(path) for path in required_files if not path.is_file()]
if missing:
    raise FileNotFoundError("Required files missing (first 12):\n" + "\n".join(missing[:12]))
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError("Image folder missing: " + str(IMAGE_DIR))
image_files = sorted(path for path in IMAGE_DIR.iterdir() if path.is_file()
                     and path.suffix.lower() in (".jpg", ".jpeg", ".png", ".bmp"))
if not image_files:
    raise FileNotFoundError("Upload an image into " + str(IMAGE_DIR))
if str(DARKNET_DIR / "python") not in sys.path:
    sys.path.insert(0, str(DARKNET_DIR / "python"))
import darknet as dn

dn.lib.parse_network_cfg.argtypes = [ctypes.c_char_p]
dn.lib.parse_network_cfg.restype = ctypes.c_void_p
dn.lib.forward_region_layer_pointer_nolayer.argtypes = [ctypes.c_void_p, ctypes.POINTER(ctypes.c_float)]
dn.lib.forward_region_layer_pointer_nolayer.restype = None
dn.lib.draw_detection_python.argtypes = [ctypes.c_void_p, ctypes.c_char_p,
    ctypes.c_float, ctypes.c_float, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p, ctypes.c_char_p]
dn.lib.draw_detection_python.restype = None
dn.lib.free_network.argtypes = [ctypes.c_void_p]
dn.lib.free_network.restype = None
print("Parameters:", QNN_PARAM_DIR)
print("Image folder:", IMAGE_DIR, "—", len(image_files), "images")


## 2. Load weights and initialize the model

Wait for **Model ready**. This cell loads the CPU weights, initializes the native detection network and FPGA, and prints a unique folder for the output images. Rerunning it first releases the previous model. If initialization fails, it releases acquired resources before reporting the error.

In [ ]:
def release_model():
    global classifier, net_darknet
    try:
        if globals().get("classifier") is not None:
            classifier.deinit_accelerator()
    finally:
        classifier = None
        if globals().get("net_darknet"):
            dn.lib.free_network(net_darknet)
        net_darknet = None

release_model()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias"):
    globals().pop(variable, None)
gc.collect()
print("Loading weights...", flush=True)
conv0_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv8_weights = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-W.npy"),
                        encoding="latin1", allow_pickle=True).transpose(3, 2, 1, 0)
conv0_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv0-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
conv8_bias = np.load(str(QNN_PARAM_DIR / "tinier-yolo-conv8-bias.npy"),
                     encoding="latin1", allow_pickle=True).reshape(-1, 1)
classifier = None
net_darknet = None
RESULT_DIR = Path(tempfile.mkdtemp(prefix="06-yolo-", dir=str(NOTEBOOK_DIR)))
try:
    classifier = TinierYolo()
    net = classifier.load_network()
    out_ch, out_dim, _ = net["conv7"]["output"]
    first_ch, first_dim, _ = net["conv1"]["input"]
    if conv0_weights.shape[0] != first_ch or conv0_bias.shape != (first_ch, 1):
        raise ValueError("First-convolution weights do not match the network.")
    if conv8_weights.shape[:2] != (125, out_ch) or conv8_bias.shape != (125, 1):
        raise ValueError("Expected the supplied 20-class, 5-anchor VOC detection head.")
    conv0_bias = np.broadcast_to(conv0_bias, (first_ch, first_dim * first_dim))
    conv8_bias = np.broadcast_to(conv8_bias, (125, out_dim * out_dim))
    net_darknet = dn.lib.parse_network_cfg(str(
        QNN_PARAM_DIR / "tinier-yolo-bwn-3bit-relu-nomaxpool.cfg").encode())
    if not net_darknet:
        raise RuntimeError("Darknet failed to load its network configuration.")
    print("Initializing FPGA...", flush=True)
    classifier.init_accelerator()
except BaseException:
    release_model()
    raise
# A unique result folder avoids overwriting results from another notebook/kernel.
RESULT_INDEX = 0
detection_history = []
print("Model ready. Result files:", RESULT_DIR)


## 3. Define image processing and detection helpers

Run this cell once. It defines image letterboxing, the CPU/FPGA stages, and Darknet drawing. Images allocated by Darknet are freed after copying. The original model's axis layout and quantization are preserved. This cell does not run a detection yet.

In [ ]:
def load_detector_image(image_path):
    original = None
    letterboxed = None
    try:
        original = dn.load_image(str(image_path).encode(), 0, 0)
        if not original.data:
            raise RuntimeError("Darknet could not read " + str(image_path))
        letterboxed = dn.letterbox_image(original, 416, 416)
        if not letterboxed.data:
            raise RuntimeError("Darknet letterboxing failed.")
        pixels = np.copy(np.ctypeslib.as_array(letterboxed.data, (3, 416, 416)))
        # Preserve the original trained model's paired spatial-axis swaps.
        return np.swapaxes(pixels, 0, 2)[np.newaxis, :, :, :]
    finally:
        if letterboxed is not None and letterboxed.data:
            dn.free_image(letterboxed)
        if original is not None and original.data:
            dn.free_image(original)

def first_convolution(pixels):
    if pixels.shape != (1, 416, 416, 3):
        raise ValueError("Expected one letterboxed 416x416 RGB image.")
    values = utils.conv_layer(pixels, conv0_weights, b=conv0_bias, stride=2, padding=1)
    return utils.quantize(values.clip(0.0, 4.0) / 4.0, 3)

def fpga_layers(quantized):
    if classifier is None:
        raise RuntimeError("Model released. Rerun section 2 before inference.")
    output = classifier.get_accel_buffer(out_ch, out_dim)
    inputs = np.ascontiguousarray(classifier.prepare_buffer(quantized * 7))
    started = time.perf_counter()
    classifier.inference(inputs, output)
    call_seconds = time.perf_counter() - started
    values = classifier.postprocess_buffer(output)
    if values.size != out_ch * out_dim * out_dim or not np.all(np.isfinite(values)):
        raise ValueError("Invalid FPGA output.")
    return values, call_seconds

def last_convolution(values):
    pixels = np.swapaxes(values.reshape(out_dim, out_dim, out_ch), 0, 1)[np.newaxis]
    output = utils.conv_layer(pixels, conv8_weights, b=conv8_bias, stride=1)
    # The native region layer reads float*, so float32 and contiguous storage are required.
    output = np.ascontiguousarray(output, dtype=np.float32)
    if output.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(output)):
        raise ValueError("Invalid detection-head output.")
    return output

def draw_detections(image_path, values, threshold=0.3):
    global RESULT_INDEX
    if net_darknet is None:
        raise RuntimeError("Model released. Rerun section 2.")
    if not 0 < threshold < 1:
        raise ValueError("Detection threshold must be between 0 and 1.")
    # Defensive check also protects callers that rerun this cell independently.
    if values.dtype != np.float32 or not values.flags.c_contiguous:
        raise ValueError("Darknet input must be contiguous float32.")
    if values.shape != (125, out_dim, out_dim) or not np.all(np.isfinite(values)):
        raise ValueError("Wrong detection-head shape or non-finite values.")
    RESULT_INDEX += 1
    result_name = "detection_{:04d}".format(RESULT_INDEX)
    output_prefix = RESULT_DIR / result_name
    output_image = RESULT_DIR / (result_name + ".png")
    probabilities = RESULT_DIR / (result_name + "_probabilities.txt")
    # This legacy C helper uses 256-byte path buffers, including added suffixes.
    paths = [str(image_path), str(VOC_NAMES), str(DARKNET_DIR) + "/",
             str(output_prefix), str(probabilities)]
    if any(len(path.encode()) > 240 for path in paths):
        raise ValueError("Use a shorter notebook/image path for the legacy Darknet helper.")
    for path in (output_image, probabilities):
        if path.exists():
            path.unlink()
    dn.lib.forward_region_layer_pointer_nolayer(
        net_darknet, values.ctypes.data_as(ctypes.POINTER(ctypes.c_float)))
    dn.lib.draw_detection_python(net_darknet, paths[0].encode(), ctypes.c_float(threshold),
        ctypes.c_float(0.5), paths[1].encode(), paths[2].encode(),
        paths[3].encode(), paths[4].encode())
    if not output_image.is_file() or not probabilities.is_file():
        raise RuntimeError("Darknet did not create the detection outputs.")
    detections = []
    for line in probabilities.read_text().splitlines():
        if not line.strip():
            continue
        name, score = line.rsplit(":", 1)
        detections.append((name.strip(), float(score.strip().rstrip("%")) / 100.0))
    return output_image, sorted(detections, key=lambda item: item[1], reverse=True)

def show_detections(output_image, detections):
    for name, score in detections:
        print("{:20s} {:6.0%}".format(name, score))
    if not detections:
        print("No objects above the selected threshold. This is a valid result.")
    with Image.open(str(output_image)) as opened:
        preview = opened.convert("RGB")
    preview.thumbnail((800, 600))
    display(preview)


## 4. Run the bounded image loop

By default, process **up to five images, once each**. Change `MAX_IMAGES` to any positive number; `REPEAT = True` cycles through the folder up to that limit. Each result shows boxes, predicted classes and scores. Each image and its scores stay visible below the previous result. Every detection also has its own numbered PNG and score file in this notebook run's output folder.

The loop stops automatically. To stop early, use **Kernel → Interrupt** and wait for the current native call to return. A `finally` block releases the FPGA and Darknet network on completion, interruption or a Python exception. To run again, rerun **section 2**, then this cell. Time measurements describe image processing, not live camera FPS or detection accuracy.

In [ ]:
MAX_IMAGES = 5  # Positive limit; fewer images are easier on a small board.
REPEAT = False  # True cycles through the folder, still stopping at MAX_IMAGES.
THRESHOLD = 0.3
results = []
try:
    if not isinstance(MAX_IMAGES, int) or isinstance(MAX_IMAGES, bool) or MAX_IMAGES < 1:
        raise ValueError("MAX_IMAGES must be a positive integer.")
    if not 0 < THRESHOLD < 1:
        raise ValueError("THRESHOLD must be between 0 and 1.")
    paths = image_files[:MAX_IMAGES] if not REPEAT else [
        image_files[index % len(image_files)] for index in range(MAX_IMAGES)]
    for number, image_path in enumerate(paths, 1):
        print("Processing {}/{}: {}".format(number, len(paths), image_path.name), flush=True)
        started = time.perf_counter()
        pixels = load_detector_image(image_path)
        quantized = first_convolution(pixels)
        fpga_output, offload_seconds = fpga_layers(quantized)
        head_output = last_convolution(fpga_output)
        output_image, detections = draw_detections(image_path, head_output, THRESHOLD)
        elapsed = time.perf_counter() - started
        print("Image {}/{}: {}".format(number, len(paths), image_path.name))
        show_detections(output_image, detections)
        print("Processing time: {:.3f} s; FPGA call: {:.3f} s".format(elapsed, offload_seconds))
        results.append(dict(image=image_path.name, output=output_image,
                            detections=detections, seconds=elapsed))
except KeyboardInterrupt:
    print("Stopped by Kernel > Interrupt.")
finally:
    release_model()
    print("Accelerator and Darknet network released. Completed images:", len(results))
if results:
    print("Mean processing time: {:.3f} s/image".format(
        sum(row["seconds"] for row in results) / len(results)))
    print("Includes image loading, CPU/FPGA inference and result file I/O; excludes notebook display.")
    print("All result images and score files are in", RESULT_DIR)


## 5. Clear large arrays when finished

The loop has already released native resources. Run this cell to clear its large Python arrays too. Restart from section 2 for a new run.

In [ ]:
release_model()
for variable in ("conv0_weights", "conv8_weights", "conv0_bias", "conv8_bias",
                 "pixels", "quantized", "fpga_output", "head_output"):
    globals().pop(variable, None)
gc.collect()
print("Accelerator and Darknet network released; large arrays cleared.")

**Validation:** Python flow and error handling are checked off-board with simulated accelerator/native interfaces and the original NumPy network helpers. Actual FPGA execution, native Darknet behavior and prediction quality must still be checked on the PYNQ board.